In [1]:
import pandas as pd
import numpy as np
from hurst import compute_Hc
import antropy as ant
from tqdm import tqdm
import warnings

warnings.filterwarnings('ignore')

In [2]:
# ==============================================================================
# 1. LOAD RAW DATA
# ==============================================================================
print("Loading raw time series data for Econophysics extraction...")
df_raw = pd.read_parquet('../data/raw/X_train.parquet').reset_index()
unique_ids = df_raw['id'].unique()

print(f"Total unique series to process: {len(unique_ids)}")

Loading raw time series data for Econophysics extraction...
Total unique series to process: 10001


In [3]:
# ==============================================================================
# 2. ECONOPHYSICS FEATURE ENGINEERING (HURST & PERMUTATION ENTROPY)
# ==============================================================================
phys_features = []
errors = 0

print("Calculating Fractal Memory and Topological Chaos...")
for uid in tqdm(unique_ids, desc="Econophysics Extraction"):
    
    # Isolate and sort the current time series
    series_data = df_raw[df_raw['id'] == uid].sort_values('time')
    
    pre_vals = series_data[series_data['period'] == 0]['value'].values
    post_vals = series_data[series_data['period'] == 1]['value'].values
    
    # Base dictionary with NaNs in case a series is too short to process
    feat_dict = {
        'id': uid, 
        'phys_hurst_pre': np.nan, 'phys_hurst_post': np.nan, 'phys_hurst_diff': np.nan,
        'phys_perm_entropy_pre': np.nan, 'phys_perm_entropy_post': np.nan, 'phys_perm_entropy_diff': np.nan
    }
    
    # The Hurst exponent requires at least ~100 data points to be statistically reliable
    if len(pre_vals) < 100 or len(post_vals) < 100:
        phys_features.append(feat_dict)
        continue
        
    try:
        # --- 1. HURST EXPONENT (Long-term Memory & Fractal Dimension) ---
        # compute_Hc returns (H, c, data). We only need H (index 0).
        # We use kind='random_walk' as a robust assumption for multi-domain anomaly detection.
        H_pre, _, _ = compute_Hc(pre_vals, kind='random_walk', simplified=True)
        H_post, _, _ = compute_Hc(post_vals, kind='random_walk', simplified=True)
        
        feat_dict['phys_hurst_pre'] = H_pre
        feat_dict['phys_hurst_post'] = H_post
        feat_dict['phys_hurst_diff'] = H_pre - H_post
        
        # --- 2. PERMUTATION ENTROPY (Topological Chaos) ---
        # order=3 is standard for financial/noisy series (looks at patterns of 3 consecutive points).
        # delay=1 means we look at immediately adjacent points.
        pe_pre = ant.perm_entropy(pre_vals, order=3, delay=1, normalize=True)
        pe_post = ant.perm_entropy(post_vals, order=3, delay=1, normalize=True)
        
        feat_dict['phys_perm_entropy_pre'] = pe_pre
        feat_dict['phys_perm_entropy_post'] = pe_post
        feat_dict['phys_perm_entropy_diff'] = pe_pre - pe_post
        
    except Exception as e:
        # Failsafe: If a series is perfectly flat (zero variance), Hurst math will fail.
        errors += 1
        pass
        
    phys_features.append(feat_dict)

print(f"\nExtraction complete. Failed/Skipped series: {errors} / {len(unique_ids)}")

Calculating Fractal Memory and Topological Chaos...


Econophysics Extraction: 100%|██████████| 10001/10001 [05:57<00:00, 28.01it/s]


Extraction complete. Failed/Skipped series: 0 / 10001


In [4]:
# ==============================================================================
# 3. POST-PROCESSING & PERSISTENCE
# ==============================================================================
df_phys = pd.DataFrame(phys_features).set_index('id')

# Impute residual NaNs with the dataset median to preserve downstream compatibility
df_phys = df_phys.fillna(df_phys.median())

save_path = '../data/processed/features_econophysics.parquet'
df_phys.to_parquet(save_path)

print(f"\nEconophysics Features successfully saved to '{save_path}'!")
display(df_phys.head())


Econophysics Features successfully saved to '../data/processed/features_econophysics.parquet'!


,phys_hurst_pre,phys_hurst_post,phys_hurst_diff,phys_perm_entropy_pre,phys_perm_entropy_post,phys_perm_entropy_diff
id,,,,,,
0,0.161397,0.151843,0.009553,0.999839,0.995666,0.004173
1,0.301615,0.293514,0.008101,0.999525,0.999504,0.000021
2,0.205168,0.244079,-0.038911,0.999798,0.998626,0.001172
3,0.260281,0.237533,0.022748,0.998725,0.998572,0.000153
4,0.143308,0.189019,-0.045711,0.991122,0.987727,0.003395
